In [53]:
import pandas as pd
from datetime import datetime

# 1. Baca data tugas
df = pd.read_csv('tasks.csv')
df

,name,difficulty,weight,deadline
0,Essay,4,8,12/5/2025
1,Presentation,3,6,12/1/2025
2,Reminder,1,3,12/20/2025


In [54]:
# 2. Ubah deadline jadi datetime
# format '%m/%d/%Y' karena bentuknya (MM/DD/YYYY)
df['deadline'] = pd.to_datetime(df['deadline'], format='%m/%d/%Y')

# 3. Hitung berapa hari lagi sampai deadline
today = datetime.now()
df['days_until_deadline'] = (df['deadline'] - today).dt.days.clip(lower=0)

df

,name,difficulty,weight,deadline,days_until_deadline
0,Essay,4,8,2025-12-05,4
1,Presentation,3,6,2025-12-01,0
2,Reminder,1,3,2025-12-20,19


In [55]:
# 4. Tambahkan target(label) yang ingin dipelajari model
priority_label_map = {
    'Presentation': 100,
    'Essay': 80,
    'Reminder': 40
}

df['priority_label'] = df['name'].map(priority_label_map)
df

,name,difficulty,weight,deadline,days_until_deadline,priority_label
0,Essay,4,8,2025-12-05,4,80
1,Presentation,3,6,2025-12-01,0,100
2,Reminder,1,3,2025-12-20,19,40


In [56]:
# 5. Masukkan Library yang diperlukan
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

In [65]:
# Latih model dengan split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.33, random_state=42
)

tmp_model = DecisionTreeRegressor(random_state=42)
tmp_model.fit(X_train, y_train)

y_pred = tmp_model.predict(X_val)
print("MAE:", mean_absolute_error(y_val, y_pred))

# Model final dilatih dengan semua data 
model = DecisionTreeRegressor(random_state=42)
model.fit(X, y)

joblib.dump(model, 'task_ranker_model.pkl')

MAE: 20.0


['task_ranker_model.pkl']

In [68]:
# Mengurutkan dari skor tinggi dulu, kalau seri, deadline paling dekat
df_sorted = df.sort_values(
    by=['priority_score', 'days_until_deadline'],
    ascending=[False, True] 
)
df_sorted

,name,difficulty,weight,deadline,days_until_deadline,priority_label,priority_score
1,Presentation,3,6,2025-12-01,0,100,100.0
0,Essay,4,8,2025-12-05,4,80,100.0
2,Reminder,1,3,2025-12-20,19,40,40.0
